# 2 · A small LLM fine-tuned with QLoRA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emrtunali/yetkin/blob/main/notebooks/02_qlora.ipynb)

The base model is loaded in **4-bit NF4** and frozen; **LoRA adapters** (rank 16) on every linear layer are trained
to answer with JSON (`{"skills": [...], "knowledge": [...]}`). The loss covers only the answer tokens. Answers are
mapped back onto the sentence's tokens, so the score is computed exactly like the taggers'.

About 1 hour on a T4 (training, then generating answers for the 3,569 test sentences).

In [ ]:
# Runtime → Change runtime type → T4 GPU, then Runtime → Run all.
!pip -q install "yetkin[train,qlora,track] @ git+https://github.com/emrtunali/yetkin.git"

import json, os, time
import torch
from google.colab import drive, userdata

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/yetkin"          # results survive a disconnect here
os.makedirs(f"{OUT}/predictions", exist_ok=True)

def secret(name):
    """Optional keys from the 🔑 Secrets panel: HF_TOKEN (to publish models), WANDB_API_KEY (to log runs)."""
    try:
        return userdata.get(name)
    except Exception:
        return None

In [ ]:
# Experiment tracking: Weights & Biases when a WANDB_API_KEY secret exists, otherwise just printed.
wandb = None
if secret("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = secret("WANDB_API_KEY")
    import wandb
print("W&B logging:", "on" if wandb else "off")

In [ ]:
from yetkin.data import load, chunks
train = [c for s in load("train") for c in chunks(s, 128)]
dev, test = load("dev"), load("test")
print(f"train {len(train):,} sentences · dev {len(dev):,} · test {len(test):,}")

In [ ]:
BASE = "Qwen/Qwen2.5-1.5B-Instruct"   # also worth a run: "Qwen/Qwen2.5-0.5B-Instruct" (a size ablation)
R, ALPHA, EPOCHS, LR = 16, 32, 2, 2e-4
slug = "qlora-" + BASE.split("/")[-1].lower()

In [ ]:
from yetkin import llm
adapter = f"{OUT}/models/{slug}"
if not os.path.exists(f"{adapter}/adapter_config.json"):
    run = wandb.init(project="yetkin", name=slug, config={"base": BASE, "r": R, "alpha": ALPHA, "epochs": EPOCHS, "lr": LR}) if wandb else None
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    info = llm.qlora(BASE, train, adapter, epochs=EPOCHS, r=R, alpha=ALPHA, lr=LR,
                     on_step=(lambda h: run.log(h)) if run else None)
    info["train_minutes"] = round((time.time() - t0) / 60, 1)
    info["peak_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else None
    json.dump({k: v for k, v in info.items() if k != "history"}, open(f"{adapter}/train_info.json", "w"), indent=1)
    if run: run.finish()
    print(f"trained in {info['train_minutes']} min, peak GPU memory {info['peak_gb']} GB")
else:
    print("adapter already trained:", adapter)

In [ ]:
from yetkin.io import save_predictions
from yetkin.metrics import score
path = f"{OUT}/predictions/{slug}.jsonl"
if not os.path.exists(path):
    model, tok = llm.load(BASE, four_bit=True, adapter=adapter)
    t0 = time.time()
    pred, raw = llm.extract(model, tok, test, batch_size=32, log=print)
    per_1k = (time.time() - t0) / len(test) * 1000
    info = json.load(open(f"{adapter}/train_info.json"))
    meta = {"name": f"QLoRA · {BASE.split('/')[-1]}", "system": f"QLoRA · {BASE.split('/')[-1]}", "base": BASE, "r": R, "alpha": ALPHA,
            "epochs": EPOCHS, "lr": LR, "trainable_params": info["trainable"], "train_minutes": info["train_minutes"],
            "peak_gpu_gb": info["peak_gb"], "seconds_per_1k_sentences": round(per_1k, 1)}
    save_predictions(path, test, pred, raw=raw, meta=meta)
    print("test strict F1:", round(score([x.spans for x in test], pred)["strict_micro"]["f1"], 4))

In [ ]:
# Optional: publish the adapter (a few MB) to the Hugging Face Hub.
if secret("HF_TOKEN"):
    from huggingface_hub import HfApi
    api = HfApi(token=secret("HF_TOKEN"))
    repo = api.create_repo(f"yetkin-{slug}", exist_ok=True).repo_id
    api.upload_folder(folder_path=adapter, repo_id=repo)
    print("published:", f"https://huggingface.co/{repo}")